# ADOM Institute - Load & Performance Benchmark on Google Colab

This notebook provisions a **complete, real** deployment of the ADOM Institute
platform inside a Colab runtime, seeds a dataset sized to that runtime, and
measures the system under concurrent load.

Nothing is mocked. You get real PostgreSQL, real Redis, real Celery workers, a
real ASGI server, and real HTTP/WebSocket clients hitting real URLs.

## What gets measured

| Layer | Tool | Metric |
|---|---|---|
| HTTP API | `harness/loadgen.py` | throughput, p50/p90/p95/p99 latency, error rate, per-scenario breakdown |
| WebSockets | `harness/wsprobe.py` | handshake success, handshake latency, broadcast fan-out latency |
| PostgreSQL | `harness/dbstat.py` | buffer cache hit ratio, dead tuples, unused indexes, `EXPLAIN (ANALYZE, BUFFERS)` on the hot read paths |
| Redis | `harness/dbstat.py` | keyspace hit ratio, memory high-water mark |
| Host | `harness/scale.py` | CPU/RAM, so you can normalise results |

## Honest expectations

* A Colab standard runtime is **~12 GB RAM / 2 vCPU** (or ~4 GB / 8 vCPU on some
  images), versus the 12 GB / 2 OCPU VM in `deploy/oci/SCALING.md` that targets
  the full ~25,000-user production dataset. So Colab is a good proxy for
  **per-request cost and query behaviour**, and a *bad* proxy for absolute
  throughput. Numbers here should be read as *"cost per request on this shape of
  data"*, not as a production capacity estimate.
* Colab runtimes are shared, throttled, and can be preempted mid-run. Treat any
  single run as indicative, and compare **relative** numbers between scenarios
  rather than absolute milliseconds.
* Google may kill the runtime without warning. Write the JSON results to Drive
  if you need to keep them.

## How to use

1. Set `REPO_URL` in the next cell to your git remote (or leave it blank and
   upload a zip of the project).
2. Run every cell top to bottom. **The tunnel cell must run before the server
   cell**, because the tunnel's random hostname has to be baked into
   `ALLOWED_HOSTS`.
3. Read the results table at the end.

In [ ]:
# ------------------------------------------------------------------ config ---
REPO_URL = ""            # e.g. "https://github.com/you/adom-institute.git"
REPO_BRANCH = "main"      # branch to check out
REPO_DIR = "/content/adom-institute-main"

# "asgi"  = one Daphne process serving HTTP + WebSockets (closest to production
#           routing behaviour, easiest to run)
# "prod"  = Gunicorn (WSGI, HTTP only) + Daphne (WebSockets only), mirroring
#           docker-compose.prod.yml so you can size the split yourself
PROFILE = "asgi"

RUN_TUNNEL = True        # Cloudflare quick tunnel -> shareable HTTPS URL
TUNNEL_TARGET_PORT = 8000

RUN_SEED = True          # False to reuse an existing database
SEED_RESET = True        # wipes the load-test tables first
SEED_PASSWORD = "LoadTest!2345"

# Load profile. Empty string = auto-size to the runtime (recommended).
# Otherwise e.g. "1,5,10,20,40" for the concurrency ladder.
LADDER = ""
print("config ok")

## 1. System services

Colab has no Docker daemon, so PostgreSQL and Redis are installed natively.
`cloudflared` is a single static binary.

In [ ]:
import os, subprocess, json, re, time, signal, socket, sys, shlex
from pathlib import Path

def sh(cmd, **kw):
    """Run a shell command, streaming output. Raises on failure."""
    print(f"\n$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, check=True, text=True,
                          capture_output=True, **kw)

def sh_ok(cmd):
    """Run a shell command, tolerate failure, return (rc, stdout, stderr)."""
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    return p.returncode, p.stdout, p.stderr

print("installing postgresql, redis, build deps ...", flush=True)
sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
   "postgresql postgresql-contrib redis-server libpq-dev curl")

# --- cloudflared (static binary) ---
cf = "/usr/local/bin/cloudflared"
if not os.path.exists(cf):
    arch = subprocess.run("uname -m", shell=True, capture_output=True,
                          text=True).stdout.strip()
    arch = "arm64" if arch in ("aarch64", "arm64") else "amd64"
    url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
    sh(f"curl -fsSL -o {cf} {url}")
    sh(f"chmod +x {cf}")
print("cloudflared:", subprocess.run(f"{cf} --version", shell=True,
      capture_output=True, text=True).stdout.strip())

In [ ]:
def start_service(name, candidates):
    """Start a systemd-less service via whatever init script exists."""
    for cmd in candidates:
        rc, out, err = sh_ok(cmd)
        if rc == 0:
            print(f"  {name}: started via `{cmd}`")
            return True
    print(f"  {name}: FAILED to start. Tried: {candidates}")
    return False

def pg_version():
    out = subprocess.run("ls /etc/postgresql", shell=True,
                         capture_output=True, text=True).stdout.split()
    return sorted(out)[-1] if out else None

PV = pg_version()
print("postgresql cluster version:", PV)

start_service("postgresql", [
    f"pg_ctlcluster {PV} main start",
    "service postgresql start",
])
start_service("redis", [
    "service redis-server start",
    "redis-server --daemonize yes",
])

def wait_for_port(host, port, timeout=60, label=""):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with socket.create_connection((host, port), timeout=2):
                print(f"  {label or host+':'+str(port)}: up after {time.time()-t0:.1f}s")
                return True
        except OSError:
            time.sleep(1)
    print(f"  {label or host}: TIMEOUT after {timeout}s")
    return False

wait_for_port("127.0.0.1", 5432, 60, "postgresql")
wait_for_port("127.0.0.1", 6379, 30, "redis")

In [ ]:
# --- create the database role and database -------------------------------
DB_NAME = "adom"
DB_USER = "adom"
DB_PASS = "adom_loadtest"

def psql(sql, dbname="postgres"):
    """Run SQL as the postgres superuser. Returns (rc, stdout, stderr)."""
    p = subprocess.run(
        ["su", "postgres", "-c",
         f'psql -v ON_ERROR_STOP=1 -d {dbname} -tAc "{sql}"'],
        capture_output=True, text=True)
    return p.returncode, p.stdout.strip(), p.stderr.strip()

def psql_or_die(sql, dbname="postgres"):
    rc, out, err = psql(sql, dbname)
    if rc != 0:
        raise SystemExit(f"psql failed: {sql!r}\n  rc={rc}\n  {err}")
    return out

role_exists = psql_or_die(f"SELECT 1 FROM pg_roles WHERE rolname='{DB_USER}'")
if not role_exists:
    psql_or_die(f"CREATE ROLE {DB_USER} LOGIN PASSWORD '{DB_PASS}'")
    print("role created:", DB_USER)
else:
    psql_or_die(f"ALTER ROLE {DB_USER} LOGIN PASSWORD '{DB_PASS}'")
    print("role already exists, password reset:", DB_USER)

db_exists = psql_or_die(f"SELECT 1 FROM pg_database WHERE datname='{DB_NAME}'")
if not db_exists:
    psql_or_die(f"CREATE DATABASE {DB_NAME} OWNER {DB_USER}")
    print("database created:", DB_NAME)
else:
    print("database already exists:", DB_NAME)

# Prove the app can actually reach it over TCP with a password. Debian's
# pg_hba.conf authenticates 127.0.0.1 with scram-sha-256, so this exercises the
# same path psycopg2 will take later -- if it fails, Django will fail too and it
# is far better to find out here.
rc, out, err = sh_ok(
    f"PGPASSWORD='{DB_PASS}' psql -h 127.0.0.1 -U {DB_USER} -d {DB_NAME} "
    f"-tAc 'SELECT version()'")
if rc != 0 or not out.strip():
    raise SystemExit(
        f"could not connect to {DB_NAME} as {DB_USER} over TCP\n"
        f"  {err or out}\n"
        "  check: pg_isready -h 127.0.0.1 ; "
        "grep -E '^(host|local)' /etc/postgresql/*/main/pg_hba.conf"
    )
print("TCP login as", DB_USER, "->", out.split(",")[0])
print("redis ping:", sh_ok("redis-cli ping")[1].strip())

## 2. Source code

Two ways in, and the cell handles both:

* **Git (recommended).** Set `REPO_URL` to your remote. If the code is not in
  Git yet, `git init && git add -A && git commit` locally, push it to a private
  repo, and paste that URL here. Note this suite does not add a remote for you.
* **Zip upload.** Use the Colab sidebar's file uploader to drop a `.zip` of the
  project into `/content`, and leave `REPO_URL` empty. The cell finds the zip,
  extracts it, and locates `manage.py` wherever it landed.

If the runtime already has the project, it is reused as-is.

In [ ]:
def find_manage_py(root: Path, depth: int = 3) -> Path | None:
    """Locate manage.py within `depth` levels, so a zip with a wrapper dir works."""
    if depth < 0:
        return None
    if (root / "manage.py").is_file():
        return root
    if not root.is_dir():
        return None
    for child in sorted(root.iterdir()):
        if child.name in (".git", "__pycache__", "node_modules"):
            continue
        hit = find_manage_py(child, depth - 1)
        if hit:
            return hit
    return None


repo = find_manage_py(Path("/content"), depth=3)

if repo is None and REPO_URL:
    sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
    repo = find_manage_py(Path(REPO_DIR), depth=2)

if repo is None:
    # A zip uploaded through the Colab sidebar lands in /content.
    zips = sorted(Path("/content").glob("*.zip"))
    if zips:
        for z in zips:
            print(f"extracting {z.name} ...")
            sh(f"unzip -q -o {shlex.quote(str(z))} -d /content")
        repo = find_manage_py(Path("/content"), depth=3)

if repo is None:
    raise SystemExit(
        "Could not find the project.\n"
        "  - set REPO_URL in the config cell to a git remote, or\n"
        "  - upload a .zip of the project using the Colab file uploader, or\n"
        "  - check it is already extracted under /content"
    )

repo = repo.resolve()
os.chdir(repo)
REPO_DIR = str(repo)
print("project root:", repo)
print("manage.py        :", (repo / "manage.py").is_file())
print("requirements.txt:", (repo / "requirements.txt").is_file())
print("colab/harness    :", sorted(p.name for p in (repo / "colab" / "harness").glob("*.py")))
print()
print(subprocess.run("git log --oneline -1", shell=True,
                     capture_output=True, text=True).stdout.strip() or "(not a git checkout)")

## 3. Python environment

An **isolated virtualenv** is used deliberately. Colab ships NumPy 2.x in the
system interpreter while `requirements.txt` pins `numpy==1.26.4`; letting pip
rewrite the system NumPy breaks unrelated runtime internals. Running every
command through `venv/bin/python` avoids that entirely.

Consequence: the notebook *kernel* is still the system Python, so all project
code must be invoked through `PY` (defined below), not via bare `python`.

In [ ]:
VENV = repo / ".venv"
if not (VENV / "bin" / "python").exists():
    sh(f"python -m venv {VENV}")
sh(f"{VENV}/bin/pip install -q --upgrade pip setuptools wheel")

print("installing project requirements ...", flush=True)
sh(f"{VENV}/bin/pip install -r {repo/'requirements.txt'}")
sh(f"{VENV}/bin/pip install aiohttp websockets psutil redis")

PY = str(VENV / "bin" / "python")
print()
print(subprocess.run(f"{PY} -c \"import django,numpy,channels;print('django',django.get_version());print('numpy',numpy.__version__)\"",
      shell=True, capture_output=True, text=True).stdout)

## 4. Django configuration

**`DEBUG=False` is not optional here, and the notebook exists partly to make
that work.** Two hard constraints in `adom/settings.py` force it:

* `settings.py:33-38` **raises `ImproperlyConfigured`** if `ALLOWED_HOSTS`
  contains `*`, `localhost`, `127.0.0.1` or `testserver` when `DEBUG=False`.
* `settings.py:89-91` appends `DebugToolbarMiddleware` when `DEBUG=True`, and
  `INTERNAL_IPS` is `['127.0.0.1']`. Since the load generator connects from
  `127.0.0.1`, a `DEBUG=True` run would measure *the debug toolbar*, not the app.

Together those mean: run with `DEBUG=False`, put **only** the tunnel hostname in
`ALLOWED_HOSTS`, and have every client send that hostname as its `Host` header
while still connecting to `127.0.0.1`. That way the benchmark exercises the real
host-allowlist path and the real security middleware, without paying tunnel
latency. `loadgen.py --host` and the WebSocket probe are both built for this.

If `RUN_TUNNEL` is False there is no hostname to allow, so the notebook falls
back to `DEBUG=True` and warns loudly - results from that mode are not
trustworthy for latency.

### Other settings worth knowing

* `STORAGES['staticfiles']` is `whitenoise.storage.CompressedManifestStaticFilesStorage`
  in both debug and non-debug mode, so `collectstatic` **must** run or
  `{% static %}` raises. It is run below.
* Redis DB allocation is split to match the intent documented at
  `settings.py:237-247` - the channel layer gets its own DB so it never
  collides with the cache or the Celery broker. The env block below keeps that
  separation; the defaults in `settings.py` would otherwise put the channel
  layer and the Celery result backend both on DB 2.
* The project logs at INFO to `logs/django.log` (`settings.py:368-389`). That
  file I/O is part of the system under test and is identical to production, so
  it is deliberately left enabled.

In [ ]:
# This block is inherited by every subprocess started later, and by the harness.
# Names must match the `config(...)` / `os.environ.get(...)` keys in
# adom/settings.py exactly -- verified against the source, do not guess.
TUNNEL_HOST = ""   # filled in by the tunnel cell, which runs before the server

env = {
    "DJANGO_SETTINGS_MODULE": "adom.settings",
    # settings.py:30 refuses the default key when DEBUG=False.
    "SECRET_KEY": "colab-loadtest-not-a-production-secret-key-0123456789",
    "DEBUG": "False",

    "DB_ENGINE": "postgresql",       # settings.py:143 requires this when not DEBUG
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASS,
    "DB_HOST": "127.0.0.1",
    "DB_PORT": "5432",
    "DB_CONN_MAX_AGE": "60",         # persistent connections, as in production

    # Four separate Redis databases, mirroring the channel-layer comment.
    "REDIS_URL": "redis://127.0.0.1:6379/0",              # cache + sessions
    "REDIS_CHANNEL_URL": "redis://127.0.0.1:6379/2",      # channels (settings default)
    "CELERY_BROKER_URL": "redis://127.0.0.1:6379/1",      # broker
    "CELERY_RESULT_BACKEND": "redis://127.0.0.1:6379/3",  # results (avoids DB 2)

    # Throttles off: DRF would otherwise measure the throttle, not the query.
    "DRF_USER_THROTTLE_RATE": "100000/min",
    "DRF_ANON_THROTTLE_RATE": "100000/min",

    # The load generator speaks plain HTTP to 127.0.0.1. The public tunnel is
    # still HTTPS, so browsers are unaffected.
    "SECURE_SSL_REDIRECT": "False",
    "SESSION_COOKIE_SECURE": "False",
    "CSRF_COOKIE_SECURE": "False",
    "SECURE_HSTS_SECONDS": "0",

    # No S3, no local LLM: Colab has neither.
    "USE_S3": "False",
    "ADOM_INSTITUTE_AI_ENABLED": "False",

    "LOADTEST_PASSWORD": SEED_PASSWORD,
}
os.environ.update(env)
print("env block ready (%d variables)" % len(env))

In [ ]:
sh(f"{PY} manage.py makemigrations --check --dry-run")
sh(f"{PY} manage.py migrate --noinput")

# PostGIS needs the extension; Colab images have it. If it is unavailable this
# prints a warning but does not abort.
sh_ok(f"{PY} manage.py migrate --noinput 2>&1 | tail -5")

sh(f"{PY} manage.py collectstatic --noinput")
print()
print("STATIC_ROOT:", subprocess.run(f"{PY} manage.py shell -c \"from django.conf import settings;print(settings.STATIC_ROOT)\"",
      shell=True, capture_output=True, text=True).stdout.strip())

## 5. Dataset

`harness/scale.py` sizes the dataset from the runtime's RAM/CPU, and
`harness/seed.py` writes it with `bulk_create`.

Seeding is the slow part (a few minutes) because it creates a few thousand users.
One password hash is computed and reused across every account: hashing 4,000
passwords with PBKDF2 would otherwise dominate the runtime. That is fine here
because authentication *cost* is not what this benchmark measures - the
permission queries are.

In [ ]:
print(subprocess.run(f"{PY} {repo/'colab/harness/scale.py'}",
      shell=True, capture_output=True, text=True).stdout)

In [ ]:
if RUN_SEED:
    flag = "--reset" if SEED_RESET else ""
    p = subprocess.run(
        f"{PY} {repo/'colab/harness/seed.py'} {flag}",
        shell=True, cwd=str(repo),
    )
    if p.returncode != 0:
        raise SystemExit("seeding failed - see the output above")
else:
    print("RUN_SEED is False; assuming the database is already populated")

In [ ]:
MANIFEST = repo / ".colab_loadtest" / "manifest.json"
manifest = json.loads(MANIFEST.read_text())
print("manifest keys:", list(manifest))
print()
print("sample logins (all share the password):")
for role, emails in manifest["tokens"]["_emails"].items():
    if emails:
        print(f"  {role:<14} {emails[0]:<38} x{len(emails)}")

## 6. Cloudflare quick tunnel

Started **before** the web server on purpose: the quick tunnel's hostname is
random, and the project refuses unknown hosts when `DEBUG=False`, so the
hostname has to be known before the server boots. Cloudflared will happily
retry against a port that is not listening yet.

If `RUN_TUNNEL` is False the notebook substitutes `127.0.0.1` and the load
tests still work locally.

In [ ]:
TUNNEL_PROC = None
TUNNEL_URL = ""

if RUN_TUNNEL:
    log = open("/tmp/cloudflared.log", "w")
    TUNNEL_PROC = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url",
         f"http://127.0.0.1:{TUNNEL_TARGET_PORT}", "--no-autoupdate"],
        stdout=log, stderr=subprocess.STDOUT)

    url = None
    for _ in range(90):
        time.sleep(1)
        text = Path("/tmp/cloudflared.log").read_text(errors="ignore")
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        if m:
            url = m.group(0)
            break
    if not url:
        print(Path("/tmp/cloudflared.log").read_text(errors="ignore")[-3000:])
        raise SystemExit("tunnel did not come up")
    TUNNEL_URL = url
    TUNNEL_HOST = url.replace("https://", "").split("/")[0]
    print("public URL:", TUNNEL_URL)

    # settings.py:33-38 rejects localhost/127.0.0.1 in ALLOWED_HOSTS when
    # DEBUG=False, so the allowlist holds the tunnel hostname and nothing else.
    # Clients reach 127.0.0.1 but send `Host: <TUNNEL_HOST>`.
    os.environ["ALLOWED_HOSTS"] = TUNNEL_HOST
    os.environ["CORS_ALLOWED_ORIGINS"] = TUNNEL_URL
    os.environ["CSRF_TRUSTED_ORIGINS"] = TUNNEL_URL
else:
    print("RUN_TUNNEL is False.")
    print("!! Falling back to DEBUG=True. settings.py:89-91 installs")
    print("!! DebugToolbarMiddleware when DEBUG is on, and INTERNAL_IPS is")
    print("!! 127.0.0.1 -- so the toolbar will be injected into every response")
    print("!! and the latency numbers will be meaningless. Enable the tunnel")
    print("!! for any run whose output you intend to trust.")
    os.environ["DEBUG"] = "True"
    os.environ["ALLOWED_HOSTS"] = "127.0.0.1,localhost,testserver"
    TUNNEL_HOST = "127.0.0.1"

print("ALLOWED_HOSTS      =", os.environ["ALLOWED_HOSTS"])
print("DEBUG              =", os.environ["DEBUG"])
print("SECURE_SSL_REDIRECT=", os.environ["SECURE_SSL_REDIRECT"])

## 7. Application processes

Order matters: Celery first, so that background work (notifications, receipt
serialisation) is not queued while the HTTP tier is being measured, then the
web tier.

For `PROFILE = "prod"`, Gunicorn owns HTTP on 8000 and Daphne owns WebSockets on
8002, matching `docker-compose.prod.yml`. `daphne` is started with
`--http-keepalive` and no `--endpoint` restrictions.

In [ ]:
PROCS = []

def start_bg(name, cmd, logfile, cwd=str(repo), env_extra=None):
    e = dict(os.environ)
    if env_extra:
        e.update(env_extra)
    f = open(logfile, "a")
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=e,
                         stdout=f, stderr=subprocess.STDOUT,
                         start_new_session=True)
    PROCS.append((name, p, logfile))
    print(f"  started {name:<10} pid={p.pid:<7} log={logfile}")
    return p

def tail(path, n=25, grep=None):
    try:
        lines = Path(path).read_text(errors="ignore").splitlines()
    except OSError:
        print(f"(no log at {path})")
        return
    for line in lines[-n:]:
        print("   ", line)

# --- celery -----------------------------------------------------------------
start_bg("celery",
         f"{PY} -m celery -A adom worker --loglevel=info --concurrency=2 "
         f"--without-gossip --without-mingle --without-heartbeat",
         "/tmp/celery.log")
time.sleep(6)
tail("/tmp/celery.log", 15)

In [ ]:
if PROFILE == "prod":
    # Gunicorn: 2 workers x 4 threads, matching the production compose file.
    start_bg("gunicorn",
             f"{PY} -m gunicorn adom.wsgi:application "
             f"--bind 0.0.0.0:{TUNNEL_TARGET_PORT} --workers 2 --threads 4 "
             f"--timeout 120 --access-logfile - --error-logfile -",
             "/tmp/gunicorn.log")
    start_bg("daphne",
             f"{PY} -m daphne -b 0.0.0.0 {TUNNEL_TARGET_PORT + 2} adom.asgi:application",
             "/tmp/daphne.log")
    BASE_URL = f"http://127.0.0.1:{TUNNEL_TARGET_PORT}"
    WS_BASE_URL = f"ws://127.0.0.1:{TUNNEL_TARGET_PORT + 2}"
else:
    start_bg("daphne",
             f"{PY} -m daphne -b 0.0.0.0 {TUNNEL_TARGET_PORT} adom.asgi:application",
             "/tmp/daphne.log")
    BASE_URL = f"http://127.0.0.1:{TUNNEL_TARGET_PORT}"
    WS_BASE_URL = BASE_URL.replace("http://", "ws://")

print()
print("BASE_URL    =", BASE_URL)
print("WS_BASE_URL =", WS_BASE_URL)
if TUNNEL_URL:
    print("public      =", TUNNEL_URL)

### Smoke test

Poll until the server answers, then confirm the real endpoints return real
data. If a request 500s here, the load test results are worthless, so this cell
raises.

In [ ]:
# Wait for the server, then prove the real endpoints return real data with a
# real token. If anything here fails, the load numbers are worthless, so this
# cell raises rather than continuing.

def http_code(path, token=None):
    cmd = ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
           "-H", f"Host: {TUNNEL_HOST}"]
    if token:
        cmd += ["-H", f"Authorization: Token {token}"]
    cmd += [f"{BASE_URL}{path}"]
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

ok = False
for _ in range(90):
    if http_code("/accounts/login/") in ("200", "301", "302"):
        ok = True
        break
    time.sleep(1)
if not ok:
    tail("/tmp/daphne.log", 60)
    raise SystemExit("server never became ready")

TOK = {k: v[0] for k, v in manifest["tokens"].items() if not k.startswith("_") and v}
print("token roles in manifest:", ", ".join(sorted(TOK)))
print()

checks = [
    ("/accounts/login/",              None,          "200"),
    ("/api/v1/students/?page_size=5", "super_admin", "200"),
    ("/api/v1/attendance/?page_size=5", "super_admin", "200"),
    ("/api/v1/fees/student-fees/?page_size=5", "accountant", "200"),
    ("/api/v1/notifications/?page_size=5", "student", "200"),
    ("/api/v1/students/?search=ST",   "super_admin", "200"),
]
bad = 0
for path, role, want in checks:
    code = http_code(path, TOK.get(role))
    okmark = "ok " if code == want else "BAD"
    if code != want:
        bad += 1
    print(f"  {okmark} {path:<44} role={role or '-':<12} -> {code} (want {want})")

if bad:
    raise SystemExit(f"{bad} smoke check(s) failed; check /tmp/daphne.log")

### The two Host headers

Note the shape of the calls above: the socket goes to `127.0.0.1`, but the
`Host` header is the tunnel hostname. That is required by
`settings.py:33-38`, and it means the benchmark exercises the same
`DisallowedHost`/CSRF-origin code path a real browser hits, without the tunnel
in the request path.

## 8. HTTP load test

`harness/loadgen.py` uses a **closed-loop** model: each virtual user sends a
request, waits for the full response, then sends the next. This is what a
browser does and it is the only model that makes p99 latency meaningful under
saturation.

* Weights are in `harness/scenarios.py`, drawn from the real URL conf: the
  teacher-scoped reads (the expensive `DISTINCT` join chain), the
  accountant fee views, the paginated student list, and the attendance writes.
* The driver sends `Host: <tunnel hostname>` while connecting to `127.0.0.1`, so
  the benchmark exercises the real host-allowlist path without paying tunnel
  latency.
* Tokens come straight from the seed manifest, so no login cost is measured.

In [ ]:
ladder_arg = f"--concurrency {LADDER}" if LADDER else ""
cmd = (f"{PY} {repo/'colab/harness/loadgen.py'} "
       f"--manifest {shlex.quote(str(MANIFEST))} "
       f"--base-url {BASE_URL} "
       f"--host-header {TUNNEL_HOST} "
       f"--duration 10 --warmup 5 {ladder_arg} "
       f"--out {repo/'colab_loadtest/http_results.json'}")
print("$", cmd, "\n")
p = subprocess.run(cmd, shell=True, cwd=str(repo))
if p.returncode != 0:
    raise SystemExit("load test failed")

## 9. WebSocket probe

Channels authenticates with a **session cookie**, not a DRF token, so the probe
performs a real form login (CSRF token and all) once per virtual user before
opening the socket. It then measures handshake latency and, for chat, the
fan-out latency of a single broadcast to every connected client - the operation
a school actually feels when a teacher submits attendance.

Note: the chat probe writes `Message` rows by design, because `ChatConsumer`
persists inbound messages.

In [ ]:
cmd = (f"{PY} {repo/'colab/harness/wsprobe.py'} "
       f"--manifest {shlex.quote(str(MANIFEST))} "
       f"--base-url {BASE_URL} "
       f"--ws-base-url {WS_BASE_URL} "
       f"--notifications 25 --chat 25 "
       f"--out {repo/'colab_loadtest/ws_results.json'}")
print("$", cmd, "\n")
# No --origin: ALLOWED_HOSTS holds only the tunnel hostname, so an Origin of
# http://127.0.0.1 would be rejected by AllowedHostsOriginValidator. A missing
# Origin is explicitly permitted (adom/asgi.py:24), and the browser path via
# the tunnel still exercises the full validator.
p = subprocess.run(cmd, shell=True, cwd=str(repo))

## 10. WebSocket results

In [ ]:
ws_path = RESULTS / "ws_results.json"
if ws_path.exists():
    ws = json.loads(ws_path.read_text())
    print(f"{'probe':<18}{'clients':>8}{'connected':>10}{'failed':>8}{'ok%':>7}"
          f"{'c-p50':>9}{'c-p95':>9}{'msgs':>7}{'fanout':>10}")
    print("-" * 86)
    for w in ws:
        print(f"{w['name']:<18}{w['clients']:>8}{w['connected']:>10}{w['failed']:>8}"
              f"{w['connect_success_pct']:>7.1f}{w['connect_p50_ms']:>9.1f}"
              f"{w['connect_p95_ms']:>9.1f}{w['messages_received']:>7}{w['fanout_p50_ms']:>10.1f}")
        for e in w["errors"]:
            print(f"      ! {e}")
    print()
    print("c-p50/c-p95 are handshake latencies. `fanout` is the time for one")
    print("chat broadcast to reach every connected client -- the number that")
    print("maps to \"how fast does a parent see the attendance alert\".")
else:
    print("no ws_results.json - run the WebSocket probe cell first")

## 11. Database and cache diagnostics

Run this **after** the load test. `idx_scan = 0` and the buffer-cache hit ratio
are only meaningful once the queries have actually executed.

In [ ]:
cmd = (f"{PY} {repo/'colab/harness/dbstat.py'} "
       f"--repo {shlex.quote(str(repo))} "
       f"--redis-url redis://127.0.0.1:6379/0 "
       f"--out {repo/'colab_loadtest/db_report.json'}")
print("$", cmd, "\n")
p = subprocess.run(cmd, shell=True, cwd=str(repo))

## 12. HTTP results in detail

The tables below are built from the JSON the harness wrote, so you can
re-render them without re-running the load.

In [ ]:
RESULTS = repo / "colab_loadtest"
http = json.loads((RESULTS / "http_results.json").read_text())

scale_info = http.get("scale") or {}
print("target      :", http["target"])
print("Host header :", http["host_header"])
print("per level   :", http["duration"], "s after warmup")
print("levels      :", [s["concurrency"] for s in http["steps"]])
print()
print("dataset under test")
print("  tier            ", scale_info.get("tier"))
print("  runtime         ", scale_info.get("cpu"), "vCPU /", scale_info.get("ram_gb"), "GB")
print("  institutions    ", scale_info.get("institutions"), "x",
      scale_info.get("academic_years"), "academic years")
print("  classes         ", (scale_info.get("derived") or {}).get("classes_total"))
print("  students        ", scale_info.get("students"))
print("  teachers        ", scale_info.get("teachers"))
print("  parents         ", scale_info.get("parents"))
print("  attendance rows ", (scale_info.get("derived") or {}).get("attendance_rows"))
print("  exam results    ", (scale_info.get("derived") or {}).get("exam_result_rows"))

In [ ]:
# Per-concurrency progression.
print(f"{'conc':>6}{'reqs':>8}{'rps':>9}{'p50':>9}{'p90':>9}{'p95':>9}{'p99':>9}{'max':>10}{'err%':>8}")
print("-" * 78)
for s in http["steps"]:
    print(f"{s['concurrency']:>6}{s['requests']:>8}{s['rps']:>9.1f}{s['p50_ms']:>9.1f}"
          f"{s['p90_ms']:>9.1f}{s['p95_ms']:>9.1f}{s['p99_ms']:>9.1f}{s['max_ms']:>10.1f}"
          f"{s['error_rate_pct']:>8.2f}")
print()
print("Read the shape, not the absolute numbers: p99 growing faster than p50 is")
print("queueing, and rps flattening while latency climbs is the knee.")

In [ ]:
# Per-scenario breakdown from the heaviest level, which is the interesting one.
heaviest = max(http["steps"], key=lambda s: s["concurrency"])
scen = [s for s in heaviest["scenarios"] if s["count"]]
print(f"breakdown at concurrency {heaviest['concurrency']} "
      f"({heaviest['wall_seconds']}s, {heaviest['requests']} requests)\n")
print(f"{'scenario':<30}{'role':<13}{'n':>6}{'err%':>7}"
      f"{'p50':>8}{'p95':>9}{'p99':>9}{'KB/s':>8}  statuses")
print("-" * 108)
for s in sorted(scen, key=lambda x: -x["p95_ms"]):
    st = ",".join(f"{k}:{v}" for k, v in list(s["status_counts"].items())[:4])
    print(f"{s['name'][:29]:<30}{s['role'][:12]:<13}{s['count']:>6}"
          f"{s['error_rate_pct']:>7.2f}{s['p50_ms']:>8.1f}{s['p95_ms']:>9.1f}"
          f"{s['p99_ms']:>9.1f}{s['kb_per_s']:>8.1f}  {st}")

In [ ]:
scen = [s for s in max(http["steps"], key=lambda s: s["concurrency"])["scenarios"]
        if s["count"]]
if not scen:
    print("no scenarios recorded")
else:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    rows = sorted(scen, key=lambda s: -s["p95_ms"])
    names = [s["name"][:34] for s in rows]
    p95 = [s["p95_ms"] for s in rows]
    bad = [s["error_rate_pct"] > 5 for s in rows]

    fig, ax = plt.subplots(figsize=(10, max(3, 0.42 * len(rows))))
    ax.barh(names[::-1], p95[::-1],
            color=["#c0392b" if b else "#2c7fb8" for b in bad[::-1]])
    ax.set_xlabel("p95 latency (ms)")
    ax.set_title("Scenarios by p95 latency (red = more than 5% errors)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

    errs = [s for s in rows if s["errors"]]
    if errs:
        print("scenarios returning errors -- check these first:\n")
        for s in errs[:6]:
            print(f"  {s['name']}  {s['errors']}/{s['count']} errors  "
                  f"statuses={s['status_counts']}")
            for sample in s["error_samples"][:2]:
                print(f"      {sample}")
        print()
    else:
        print("no scenario produced an error")

In [ ]:
# Persist everything to Drive so a preemption does not lose the run.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    out = Path("/content/drive/adom_loadtest_results")
    out.mkdir(parents=True, exist_ok=True)
    for f in RESULTS.glob("*.json"):
        (out / f.name).write_text(f.read_text())
    print("copied to", out)
except Exception as exc:
    print("Drive save skipped:", exc)

## 13. Interpreting the numbers

The queries worth arguing about, all of which `dbstat.py` prints a plan for:

* **Teacher-scoped student list** - `accounts/permissions.py` joins
  `students -> classes -> class_schedules -> teachers` and applies `DISTINCT`.
  With 40 classes x 35 weekly periods there are 1,400 schedule rows, so the join
  fan-out is real. Expect a `Hash Join` plus a `Sort` for `DISTINCT`; if the
  sort spills to disk (`external merge`) then `work_mem` is too small, and if
  the plan is a `Seq Scan` on `timetable_class_schedule`, the teacher index is
  not being used.
* **Attendance summary** - `COUNT ... FILTER` over the whole
  `attendance_attendance` table. The existing index is
  `(student, date, status)`, so a filter on `date` alone cannot use it. This is
  the single highest-value index to add: `(date, status)`.
* **Student search** - `icontains` on `student_id` forces a sequential scan no
  matter what, because there is no trigram index. If that search is on a hot
  user path in production, `pg_trgm` is the fix.
* **Overdue fees** - already has `(payment_status, due_date)`, so this one
  should be an index scan. If it is not, the predicate and the index disagree.

### Caveats specific to Colab

* A quick tunnel is rate-limited and adds nothing useful to a benchmark; it is
  here so you can *look* at the app from a phone, not to measure through.
* Cold caches dominate the first minutes. The warmup phase exists for this, but
  if p99 looks absurd, check the raw samples before believing it.
* The vCPU count on Colab does not scale linearly with reported cores, and the
  host is shared with other tenants. Run any comparison at least three times.

## 14. Teardown

Killing the background processes without re-running the whole notebook:

```python
for name, p, log in PROCS:
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except Exception:
        pass
```

In [ ]:
# Optional: stop everything (commented out so results stay inspectable).
STOP = False
if STOP:
    for name, p, logfile in PROCS:
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGTERM)
            print("stopped", name)
        except Exception as exc:
            print("could not stop", name, exc)
    if TUNNEL_PROC:
        TUNNEL_PROC.terminate()
else:
    print("servers still running:")
    for name, p, logfile in PROCS:
        print(f"  {name:<10} pid={p.pid} alive={p.poll() is None}  log={logfile}")
    if TUNNEL_URL:
        print(f"  tunnel     {TUNNEL_URL}")
    print()
    print("  tail logs:  tail -f /tmp/daphne.log")
    print("  full run:   tail -n 200 /tmp/celery.log /tmp/daphne.log /tmp/gunicorn.log")